# SEC Proxy Ingestion Pipeline — ConnectOne Bancorp (CNOB) DEF 14A 2025


In [ ]:
import sys, requests, json
from pathlib import Path
from IPython.display import display
import pandas as pd
sys.path.insert(0, str(Path("..").resolve()))

from ingestion.metadata_model import DocumentMetadata
from ingestion.sec_html_parser import SECHTMLParser
from ingestion.sec_chunker import SECChunker

FILING_URL = "https://www.sec.gov/Archives/edgar/data/712771/000143774925011656/cnob20240411_def14a.htm"
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
cache_path = DATA_DIR / "712771_000143774925011656.html"
if cache_path.exists():
    raw_html = cache_path.read_text(encoding="utf-8")
    print(f"Loaded from cache: {cache_path} ({len(raw_html):,} chars)")
else:
    headers = {"User-Agent": "YaNa Research yashar@example.com"}
    resp = requests.get(FILING_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    raw_html = resp.text
    cache_path.write_text(raw_html, encoding="utf-8")
    print(f"Downloaded and cached: {cache_path} ({len(raw_html):,} chars)")


## Filing Metadata


In [ ]:
from datetime import date
metadata = DocumentMetadata(
    document_id="712771_000143774925011656",
    cik="712771",
    company_name="ConnectOne Bancorp, Inc.",
    form_type="DEF 14A",
    filing_date=date(2025, 4, 11),
    accession_number="0001437749-25-011656",
    source_url=FILING_URL,
    raw_html_path=str(cache_path),
)
print(metadata.model_dump_json(indent=2))


## Step 1: Parse HTML → Typed Blocks


In [ ]:
parser = SECHTMLParser()
blocks = parser.parse(raw_html, metadata)
print(f"Total blocks extracted: {len(blocks)}")


In [ ]:
from collections import Counter
type_counts = Counter(type(b).__name__ for b in blocks)
df_types = pd.DataFrame(type_counts.items(), columns=["BlockType", "Count"]).sort_values("Count", ascending=False)
display(df_types)


## Step 2: Inspect Sample Blocks


In [ ]:
headings = [b for b in blocks if type(b).__name__ == "HeadingBlock"]
for h in headings[:5]:
    print(f"[{h.detection_method}] level={h.level} | section_id={h.section_id} | text={h.text[:80]}")


In [ ]:
tables = [b for b in blocks if type(b).__name__ == "TableBlock"]
print(f"Total tables: {len(tables)}")
if tables:
    t = tables[0]
    print(f"\nTable ID: {t.id}")
    print(f"Rows: {len(t.rows)}, Header rows: {t.header_row_count}, Merged cells: {t.has_merged_cells}")
    print(f"Footnotes: {t.footnotes}")
    print(f"\nLinearized (first 300 chars):\n{t.linearized_text[:300]}")


In [ ]:
xbrl_blocks = [b for b in blocks if type(b).__name__ == "XBRLTaggedBlock"]
print(f"XBRL-tagged blocks: {len(xbrl_blocks)}")
for x in xbrl_blocks[:3]:
    print(f"\n  text[:80]: {x.text[:80]}")
    for ann in x.xbrl_tags:
        print(f"  concept: {ann.concept_name}, contextRef: {ann.context_ref}")


## Step 3: Chunk Blocks


In [ ]:
chunker = SECChunker()
chunks = chunker.chunk_blocks(blocks, metadata)
print(f"Total chunks: {len(chunks)}")


In [ ]:
section_counts = Counter(c.section_id for c in chunks)
df_sections = pd.DataFrame(section_counts.items(), columns=["SectionID", "ChunkCount"]).sort_values("ChunkCount", ascending=False)
display(df_sections.head(15))


In [ ]:
token_counts = [c.token_count for c in chunks]
print(f"Min tokens: {min(token_counts)}")
print(f"Max tokens: {max(token_counts)}")
print(f"Mean tokens: {sum(token_counts)/len(token_counts):.1f}")
assert max(token_counts) <= 600, f"CONSTRAINT VIOLATED: chunk has {max(token_counts)} tokens"
print("✓ All chunks within 600-token limit")


In [ ]:
for c in chunks[:3]:
    print(c.citation_string)


## Step 4: Export to CSV


In [ ]:
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rows = [
    {
        "chunk_id": c.id,
        "chunk_index": c.chunk_index,
        "document_id": c.document_id,
        "section_id": c.section_id,
        "token_count": c.token_count,
        "citation_string": c.citation_string,
        "text_preview": c.text[:120].replace("\n", " "),
    }
    for c in chunks
]
df_chunks = pd.DataFrame(rows)
df_chunks.to_csv(OUTPUT_DIR / "chunks_cnob_m0.csv", index=False)
print(f"Exported {len(df_chunks)} chunks to output/chunks_cnob_m0.csv")
display(df_chunks.head(5))


## M0 Assertions — Evidence Gate


In [ ]:
assert len(blocks) > 50, f"Expected >50 blocks, got {len(blocks)}"
assert len(tables) >= 5, f"Expected >=5 tables in CNOB DEF 14A, got {len(tables)}"
assert len(headings) >= 10, f"Expected >=10 section headings, got {len(headings)}"
assert len(chunks) > 50, f"Expected >50 chunks, got {len(chunks)}"
assert all(c.citation_string for c in chunks), "Some chunks missing citation_string"
assert all(c.token_count > 0 for c in chunks), "Some chunks have zero token count"
# Every chunk's text must be a substring of original HTML or linearized table text
for c in chunks:
    if c.table_json is None:
        assert c.text in raw_html, f"Chunk text not found in source HTML: {c.text[:60]}"
print("✓ All M0 assertions passed")
